# Exercise 1 · Check every reply against a schema

You will sort model replies into passes and failures, fixing none.

**Goal:** Check model replies against a schema in your own code, and count each bad one as a failure.

**Why it matters:** Code that saves a badly shaped reply saves bad data; your own check stops it first.

**From the slides:** Slide 4, "A schema guarantees shape, not content; it can cost accuracy"; Slide 5, "Every field is required, and only the amount may be null"; Slide 7, "Validate every reply in your own code anyway".

**Before you start:**
- JSON is text that holds data; `json.loads` turns it into a Python dict, with `None` for `null`.
- A JSON Schema is a dict that describes good data; the `jsonschema` library checks data against it.
- A function can raise an error, such as `ValueError`, to report a problem.

**What the check measures:** the last cell makes up good and bad replies to test your schema. It then checks that `check_reply` passes replies 1 and 4, and raises `ValueError` for the other six and for `NEW_REPLY` (Step 4), also with order id `XA1042` or `AB042`.

**Time:** about 25 minutes.

**Done when:** the last cell prints `Done:`.

## The situation

A small online shop sends each customer message to an AI model and asks for four fields, as JSON:

- `order_id`: text such as `"A1042"` (the letter A, then four digits)
- `issue`: one of `"late"`, `"damaged"`, `"wrong_item"`, `"other"`
- `refund_amount`: the refund the customer asks for, as a number, or `null` (no value) when the message names none
- `urgent`: `true` or `false`

Your check runs before the fields are saved. A bad reply counts as a failure, never fixed or guessed.

## Step 1 · Load the sample replies

Run the cell. It prints 8 sample replies: simulated text, the way a model returns it.

Expect 8 lines, from `reply 1:` to `reply 8:`.

In [ ]:
# Step 1: load and print the sample replies
import json

import jsonschema

REPLIES = {  # Sample replies (simulated)
    "reply 1": '{"order_id": "A1042", "issue": "late", "refund_amount": null, "urgent": false}',
    "reply 2": '{"order_id": "A3307", "issue": "wrong_item", "refund_amount": "about 20 dollars", "urgent": false}',
    "reply 3": '{"order_id": "A1185", "issue": "late", "refund_amount": null}',
    "reply 4": '{"order_id": "A2210", "issue": "damaged", "refund_amount": 35.5, "urgent": true}',
    "reply 5": '{"order_id": "A4521", "issue": "other", "refund_amount": null, "urgent": false, "confidence": 0.9}',
    "reply 6": '{"order_id": "A2093", "issue": "broken", "refund_amount": 12, "urgent": true}',
    "reply 7": "Sorry, I can't help with that request.",  # a refusal
    "reply 8": '{"order_id": "A3876", "issue": "damaged", "refund_amou',  # cut off at the model's length limit
}
for name, text in REPLIES.items():
    print(f"{name}: {text}")

## Step 2 · Complete the schema

Replace each `# TODO:` line in `REPLY_SCHEMA` as its comment says: `"enum"` lists the allowed values, `"required"` the fields that must be there, and `"additionalProperties": False` allows no others.

Expect `Required:` to list all four fields, then `Extra fields allowed: False`.

<details><summary>Hint</summary>

Each field gets a dict, like `{"type": "string"}` or `{"enum": ["a", "b"]}`. A list of types allows either one: `["number", "null"]`. Python writes `False`, not `false`.

</details>

<details><summary>Show the answer</summary>

```python
REPLY_SCHEMA = {
    "type": "object",
    "properties": {
        "order_id": {"type": "string"},
        "issue": {"enum": ["late", "damaged", "wrong_item", "other"]},
        "refund_amount": {"type": ["number", "null"]},
        "urgent": {"type": "boolean"},
    },
    "required": ["order_id", "issue", "refund_amount", "urgent"],
    "additionalProperties": False,
}
```

Why: each field must be there with its type, and null lets a reply say "no amount" instead of inventing one.

</details>

In [ ]:
# Step 2: describe a good reply, then print what the schema asks for
REPLY_SCHEMA = {
    "type": "object",
    "properties": {
        "order_id": {"type": "string"},
        "issue": {"enum": ["late", "damaged", "wrong_item", "other"]},
        "refund_amount": {"type": ["number", "null"]},
        "urgent": {"type": "boolean"},
    },
    "required": ["order_id", "issue", "refund_amount", "urgent"],
    "additionalProperties": False,
}
# Stop here if the schema is not valid:
jsonschema.Draft202012Validator.check_schema(REPLY_SCHEMA)
print("Required:", REPLY_SCHEMA.get("required", []))
print("Extra fields allowed:", REPLY_SCHEMA.get("additionalProperties", True))

## Step 3 · Complete check_reply and count the failures

`check_reply` returns a reply's fields when the text is JSON that matches `REPLY_SCHEMA`; otherwise it raises `ValueError` with the reason. It never fixes a reply. Replace its two `# TODO:` lines.

Expect replies 1 and 4 to pass, the others to fail with a reason, and `Failures: 6 of 8 replies`. If reply 1 or 4 fails, fix the Step 2 rule its reason names, then run Steps 2 and 3 again. If fewer fail, a bad reply got through.

<details><summary>Hint</summary>

Each `# TODO:` comment names the function to call; `jsonschema.validate` takes the data first. The `except` lines turn its errors into `ValueError`.

</details>

<details><summary>Show the answer</summary>

Write `fields = json.loads(text)` on the first `# TODO:` line, and `jsonschema.validate(fields, REPLY_SCHEMA)` in place of `pass`. Why: one reads the reply and the other checks its shape; neither changes it.

</details>

In [ ]:
# Step 3: check every reply and count the failures
def check_reply(text: str) -> dict:
    """Return the reply's fields if it is JSON matching REPLY_SCHEMA; else raise ValueError."""
    try:
        fields = json.loads(text)
    except json.JSONDecodeError:
        raise ValueError("the reply is not JSON")
    try:
        jsonschema.validate(fields, REPLY_SCHEMA)
    except jsonschema.ValidationError as error:
        raise ValueError(error.message)
    return fields


def verdict(text: str) -> str:
    """Return "passes", or "fails: " and the reason from check_reply."""
    try:
        check_reply(text)
        return "passes"
    except ValueError as reason:
        return f"fails: {reason}"


failures = 0
for name, text in REPLIES.items():
    result = verdict(text)
    print(f"{name}: {result}")
    if result != "passes":
        failures += 1
print(f"Failures: {failures} of {len(REPLIES)} replies")

## Step 4 · Add a rule for the order id

Your schema says only that `order_id` is a string, so `NEW_REPLY` passes although its order id, `"A10425"`, has five digits. Set `ORDER_ID_PATTERN` to a pattern (a regular expression, which describes text) for A and then four digits.

Expect `Before your rule: passes`, then `With your rule: fails: 'A10425' does not match` followed by your pattern.

<details><summary>Hint</summary>

In a pattern, `^` marks the start, `A` is the letter, `[0-9]{4}` is four digits, and `$` marks the end. Without them, `A[0-9]{4}` finds `A1042` inside `A10425`.

</details>

<details><summary>Show the answer</summary>

`ORDER_ID_PATTERN = "^A[0-9]{4}$"`. Why: `^` and `$` make the whole id match, not just part of it.

</details>

In [ ]:
# Step 4: check one more sample reply before and after your order id rule
NEW_REPLY = '{"order_id": "A10425", "issue": "late", "refund_amount": null, "urgent": false}'
ORDER_ID_PATTERN = "^A[0-9]{4}$"
REPLY_SCHEMA["properties"]["order_id"] = {"type": "string"}  # Step 2's rule, no pattern
print("Before your rule:", verdict(NEW_REPLY))
REPLY_SCHEMA["properties"]["order_id"]["pattern"] = ORDER_ID_PATTERN
print("With your rule:", verdict(NEW_REPLY))

## Check your work

Run the cell. Expect a `Done:` line. If it prints `Not yet:`, fix what it names and run the cells from that step down.

In [ ]:
# Check your work: test your schema and check_reply
GOOD = {"order_id": "A1042", "issue": "late", "refund_amount": None, "urgent": False}
MUST_FAIL = {  # made-up bad replies
    'issue is "broken"': {**GOOD, "issue": "broken"},
    'refund_amount is "20"': {**GOOD, "refund_amount": "20"},
    "urgent is null": {**GOOD, "urgent": None},
    **{f"{f} is missing": {k: v for k, v in GOOD.items() if k != f} for f in GOOD},
    "a field is added": {**GOOD, "note": "hi"},
}
MUST_PASS = [{**GOOD, "issue": i} for i in ("late", "damaged", "wrong_item", "other")]
MUST_PASS.append({**GOOD, "refund_amount": 35.5})


def problem() -> str:
    """Return the first thing to fix, or "" if none."""
    validator = jsonschema.Draft202012Validator(REPLY_SCHEMA)
    for reply in MUST_PASS:
        error = jsonschema.exceptions.best_match(validator.iter_errors(reply))
        if error:
            return (f"your schema fails a good reply: {error.message}. "
                    "Fix that rule in Step 2 or 4.")
    for flaw, reply in MUST_FAIL.items():
        if validator.is_valid(reply):
            return f"your schema passes a made-up reply where {flaw}. Fix that in Step 2."
    for name, text in REPLIES.items():
        good = name in ("reply 1", "reply 4")
        if (verdict(text) == "passes") != good:
            return (f"{name} {verdict(text)}, but should {'pass' if good else 'fail'}. "
                    "check_reply must validate each reply and fix none (Step 3).")
    for order_id in ("A10425", "XA1042", "AB042"):
        if verdict(NEW_REPLY.replace("A10425", order_id)) == "passes":
            return (f"your rule passes order id {order_id}. "
                    'Run Step 4 with "^A[0-9]{4}$" (^ and $ included).')
    return ""


try:
    found = problem()
except Exception as error:
    reason = str(error).strip().split("\n")[0].rstrip(":")
    found = (f"the check stopped with {type(error).__name__}: {reason}. "
             "Run all cells from the top and fix the first cell that stops.")
print(f"Not yet: {found}" if found else
      "Done: replies 1 and 4 pass; the other six and NEW_REPLY fail, none fixed.")

## What you learned

- A schema checks only the shape rules you write: your pattern stops `"A10425"`, but `"A9999"` passes even if no such order exists (Slide 4).
- Every field is required, and only `refund_amount` may be null, so a reply never has to invent an amount (Slide 5).
- Check every reply in your own code; count each bad one as a failure, never fixed or guessed (Slide 7).

**Reflect:**
- A teammate wants `refund_amount` to accept text so that fewer replies fail. What would the shop then save?